# Score-CAM vs GradCAM / LayerCAM — PanNuke Comparison

This notebook adds **Score-CAM** [Wang et al., 2020] as a gradient-free baseline
and compares it against the four gradient-based methods already evaluated.

## Why Standard Score-CAM only (no FPN-ScoreCAM)

Score-CAM measures each channel's causal contribution by masking the input image
with the upsampled activation map for that channel, running a full forward pass,
and measuring the change in class confidence. DenseNet169's `denseblock4` has
**1,664 channels**, so Standard Score-CAM requires **1,664 forward passes per
class per image** — already ~1.66 M passes for 500 images × 2 classes.

FPN-ScoreCAM would require per-channel masking at *three independent layers*
(db1: 256, db2: 512, db4: 1,664 = 2,432 channels total). More importantly,
early-layer channel contributions in a multi-scale fusion require the full forward
pass to propagate through all downstream layers — the FPN blend cannot be applied
post-hoc as it is for gradient methods. The implementation complexity is
substantially higher and the scientific question ("does the pyramid help
Score-CAM?") is already answered by the gradient-based FPN results.
**FPN-ScoreCAM is explicitly left as future work.**

## Computational strategy
Score-CAM channels are processed in **batches of 64** to fill the GPU while
staying within 8 GB VRAM. Expected runtime: ~1–2 GPU hours for 500 images.

## What this notebook produces
- `scorecam_iou_summary.csv` — IoU@{0.25,0.40,0.50} per method × class
- `scorecam_results_500_test.csv` — per-image IoU for all 5 methods
- `scorecam_vs_gradient_comparison.csv` — side-by-side macro IoU
- Figures: IoU bar charts, heatmap, cardinality plot, factorial comparison

## 1. Imports & Environment

In [ ]:
import gc
import json
import random
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 120

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Configuration
All constants match the main PanNuke CAM notebook exactly.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR     = Path('.')
OUTPUT_ROOT  = Path('./outputs')
MODEL_PATH   = OUTPUT_ROOT / 'best_densenet169_pannuke.pth'
TEST_JSON    = OUTPUT_ROOT / 'test_predictions.json'
TEST_IMG_DIR = OUTPUT_ROOT / 'test_images'

# Results from the main CAM notebook — loaded for comparison
GRAD_RESULTS_CSV = OUTPUT_ROOT / 'layercam_comparison' / 'results_50_test.csv'

OUT_DIR  = OUTPUT_ROOT / 'scorecam_comparison'
FIG_DIR  = OUT_DIR / 'figures'
TRIP_DIR = OUT_DIR / 'triplets' / 'scorecam'
for d in [OUT_DIR, FIG_DIR, TRIP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FOLD_DIRS: Dict[int, Dict[str, Path]] = {
    fold: {
        'images': BASE_DIR / f'fold_{fold}' / 'images' / f'fold_{fold}' / 'images.npy',
        'masks' : BASE_DIR / f'fold_{fold}' / 'masks'  / f'fold_{fold}' / 'masks.npy',
    }
    for fold in [1, 2, 3]
}

# ── Dataset constants ─────────────────────────────────────────────────────────
CLASS_KEYS: List[str]    = ['Neoplastic', 'NonneoplasticEpi', 'Inflammatory', 'Connective', 'Dead']
CLASS_DISPLAY: List[str] = ['Neoplastic', 'Non-neo. Epi.', 'Inflammatory', 'Connective', 'Dead']
NUM_CLASSES  = 5
IMG_SIZE     = 256
THRESHOLD    = 0.5
IOU_THRS     = [0.25, 0.40, 0.50]
NOISE_THR    = 0.25
ALPHA_SCALE  = 0.85

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Score-CAM specific ────────────────────────────────────────────────────────
# Layer to hook for Score-CAM — same as gradient methods (denseblock4)
SCORECAM_LAYER = 'backbone.features.denseblock4'
# Number of channels processed per GPU batch.
# 64 channels × 1 masked image each = 64 forward passes per batch.
# Reduce to 32 if OOM; increase to 128 if VRAM headroom allows.
SCORECAM_BATCH = 64

# ── Colour palette (identical to main notebook) ───────────────────────────────
CLASS_COLORS_HEX = ['#E53935', '#43A047', '#1E88E5', '#FFB300', '#8E24AA']
CLASS_COLORS_F32 = [
    (int(h[1:3], 16)/255, int(h[3:5], 16)/255, int(h[5:7], 16)/255)
    for h in CLASS_COLORS_HEX
]

# All methods in the final comparison (4 gradient + 1 Score-CAM)
ALL_METHODS = ['std_gradcam', 'fpn_gradcam', 'std_layercam', 'fpn_layercam', 'scorecam']
METHOD_LABELS = {
    'std_gradcam'  : 'Standard GradCAM',
    'fpn_gradcam'  : 'FPN-GradCAM',
    'std_layercam' : 'Standard LayerCAM',
    'fpn_layercam' : 'FPN-LayerCAM',
    'scorecam'     : 'Score-CAM',
}
METHOD_COLORS = {
    'std_gradcam'  : '#90CAF9',
    'fpn_gradcam'  : '#EF9A9A',
    'std_layercam' : '#A5D6A7',
    'fpn_layercam' : '#FFE082',
    'scorecam'     : '#CE93D8',   # purple — gradient-free family
}
METHOD_MARKERS = {
    'std_gradcam'  : 'o',
    'fpn_gradcam'  : 's',
    'std_layercam' : '^',
    'fpn_layercam' : 'D',
    'scorecam'     : 'P',
}

print('Configuration loaded.')
print(f'Score-CAM layer   : {SCORECAM_LAYER}')
print(f'Score-CAM batch   : {SCORECAM_BATCH} channels/batch')
print(f'Output dir        : {OUT_DIR}')

## 3. Model

In [ ]:
class DenseNet169MultiLabel(nn.Module):
    def __init__(self, num_classes: int = 5, dropout_p: float = 0.3) -> None:
        super().__init__()
        backbone = models.densenet169(weights=None)
        in_features = backbone.classifier.in_features  # 1664
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(512, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.backbone(x))


model = DenseNet169MultiLabel(num_classes=NUM_CLASSES).to(DEVICE)
ckpt  = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded checkpoint — epoch {ckpt["epoch"]}, val AUC {ckpt["val_auc"]:.4f}')
print(f'Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f} M')

## 4. Preprocessing & Inference Utilities

In [ ]:
_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def preprocess(img_rgb: np.ndarray) -> torch.Tensor:
    """HWC uint8 → (1,3,H,W) float32 on DEVICE."""
    return _transform(img_rgb).unsqueeze(0).to(DEVICE)


@torch.no_grad()
def predict(img_rgb: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Returns (binary_labels, sigmoid_probs)."""
    logits = model(preprocess(img_rgb))
    probs  = torch.sigmoid(logits).squeeze().cpu().numpy()
    return (probs >= THRESHOLD).astype(np.uint8), probs


def compute_iou(cam: np.ndarray, gt_ch: np.ndarray, thr: float) -> float:
    gt_bin = gt_ch > 0
    if not gt_bin.any():
        return float('nan')
    pred  = cam >= thr
    inter = (pred & gt_bin).sum()
    union = (pred | gt_bin).sum()
    return float(inter) / float(union) if union > 0 else float('nan')


def resize_mask_ch(
    mask_6ch: np.ndarray, class_idx: int,
    hw: Tuple[int, int] = (IMG_SIZE, IMG_SIZE),
) -> np.ndarray:
    ch = mask_6ch[..., class_idx].astype(np.float32)
    if ch.shape != hw:
        ch = cv2.resize(ch, (hw[1], hw[0]), interpolation=cv2.INTER_NEAREST)
    return ch


def render_gt_mask(mask_6ch: np.ndarray,
                   hw: Tuple[int, int] = (IMG_SIZE, IMG_SIZE)) -> np.ndarray:
    H, W   = hw
    canvas = np.zeros((H, W, 3), dtype=np.float32)
    for ci, color in enumerate(CLASS_COLORS_F32):
        ch = resize_mask_ch(mask_6ch, ci, hw)
        canvas[ch > 0] = color
    return (canvas * 255).clip(0, 255).astype(np.uint8)


def build_cam_overlay(
    cam: np.ndarray, class_idx: int, orig: np.ndarray,
) -> np.ndarray:
    """Single-class CAM overlay on original image."""
    clean = np.where(cam >= NOISE_THR, cam, 0.0)
    r, g, b = CLASS_COLORS_F32[class_idx]
    comp_c = np.stack([clean * r, clean * g, clean * b], axis=-1)
    alpha  = (clean * ALPHA_SCALE)[..., None]
    bg     = orig.astype(np.float32) / 255.0
    out    = (comp_c * alpha + bg * (1 - alpha)).clip(0, 1)
    return (out * 255).astype(np.uint8)


print('Utilities defined.')

## 5. Score-CAM Implementation

Score-CAM [Wang et al., CVPR Workshops 2020] is a **gradient-free** attribution method.
Instead of backpropagating gradients, it measures each channel's causal contribution
by masking the input with that channel's normalised activation map and observing
the resulting change in class confidence (the *increase in confidence score*).

**Algorithm for class c at layer L with K channels:**
1. Run a forward pass to obtain activation maps $A^k \in \mathbb{R}^{h \times w}$ for $k=1\ldots K$.
2. For each channel $k$: upsample $A^k$ to input resolution, min-max normalise to $[0,1]$, multiply element-wise with the original image to produce a masked input $X_k$.
3. Run a forward pass on $X_k$ and extract the softmax score for class $c$: $s_k = \sigma(f_c(X_k))$.
4. Use $s_k$ as the channel importance weight: $L^c = \mathrm{ReLU}\bigl(\sum_k s_k \cdot A^k\bigr)$.

**Key property:** No gradient is computed. The importance weights come from
*causal intervention* (masking) rather than *gradient sensitivity*.
This means Score-CAM is immune to gradient saturation and to the
multi-label gradient contamination mechanism identified in the main study.

**Batching:** All 1,664 channel masks are stacked and processed in batches
of `SCORECAM_BATCH` to avoid OOM. The baseline forward pass (unmasked image)
produces the activation maps used for both the mask generation and the
final weighted sum.

In [ ]:
class ScoreCAMEngine:
    """Gradient-free Score-CAM for DenseNet169 multi-label classification.

    Parameters
    ----------
    model       : trained DenseNet169MultiLabel, in eval mode
    layer_path  : dotted path to the target convolutional layer
                  (default: 'backbone.features.denseblock4')
    batch_size  : number of masked forward passes per GPU batch
    """

    def __init__(
        self,
        model      : nn.Module,
        layer_path : str  = SCORECAM_LAYER,
        batch_size : int  = SCORECAM_BATCH,
    ) -> None:
        self.model      = model
        self.batch_size = batch_size
        self._acts: Optional[torch.Tensor] = None

        # Navigate to target layer
        layer = model
        for k in layer_path.split('.'):
            layer = getattr(layer, k)

        # Single forward hook — no backward hook needed
        self._handle = layer.register_forward_hook(
            lambda m, i, o: setattr(self, '_acts', o.detach())
        )

    def remove_hooks(self) -> None:
        self._handle.remove()

    @torch.no_grad()
    def generate(
        self,
        img_tensor  : torch.Tensor,   # (1, 3, H, W) normalised, on DEVICE
        img_norm    : torch.Tensor,   # same tensor — used for masking
        class_idx   : int,
    ) -> np.ndarray:                  # (H, W) float32 [0, 1]
        """Generate Score-CAM heatmap for one class.

        Args:
            img_tensor : (1,3,H,W) normalised input on DEVICE
            img_norm   : same tensor (kept separate for clarity)
            class_idx  : class index to explain

        Returns:
            cam : (H, W) float32 normalised to [0, 1]
        """
        H, W = img_tensor.shape[2], img_tensor.shape[3]

        # ── Step 1: baseline forward pass to get activation maps ──────────────
        _ = self.model(img_tensor)         # fills self._acts via hook
        acts = self._acts.squeeze(0)       # (K, h, w)
        K, h, w = acts.shape

        # ── Step 2: upsample and normalise each channel map ───────────────────
        # acts_up: (K, H, W) — each channel map upsampled to input resolution
        acts_up = F.interpolate(
            acts.unsqueeze(0),             # (1, K, h, w)
            size=(H, W),
            mode='bilinear',
            align_corners=False,
        ).squeeze(0)                       # (K, H, W)

        # Min-max normalise each channel to [0, 1] independently
        a_min = acts_up.flatten(1).min(dim=1).values.view(K, 1, 1)
        a_max = acts_up.flatten(1).max(dim=1).values.view(K, 1, 1)
        denom = (a_max - a_min).clamp(min=1e-8)
        masks = (acts_up - a_min) / denom  # (K, H, W) in [0, 1]

        # ── Step 3: masked forward passes in batches ──────────────────────────
        # Each masked image = original × channel_mask (element-wise)
        scores = torch.zeros(K, device=DEVICE)  # s_k for each channel

        for start in range(0, K, self.batch_size):
            end       = min(start + self.batch_size, K)
            batch_k   = end - start

            # Expand original image to batch: (batch_k, 3, H, W)
            img_batch = img_tensor.expand(batch_k, -1, -1, -1)  # no copy

            # Apply channel masks: (batch_k, 1, H, W) broadcast over channels
            mask_batch = masks[start:end].unsqueeze(1)          # (batch_k,1,H,W)
            masked     = img_batch * mask_batch                  # (batch_k,3,H,W)

            # Forward pass — extract sigmoid score for target class
            logits = self.model(masked)                          # (batch_k, C)
            scores[start:end] = torch.sigmoid(logits[:, class_idx])

        # ── Step 4: weighted sum of activation maps ───────────────────────────
        # scores: (K,) → reshape to (K, 1, 1) for broadcasting
        w = scores.view(K, 1, 1)                                 # (K, 1, 1)

        # acts is still on GPU at original spatial resolution (h, w)
        # Upsample once more for the weighted sum
        cam_raw = F.relu((w * acts).sum(dim=0, keepdim=True))    # (1, h, w)
        cam_up  = F.interpolate(
            cam_raw.unsqueeze(0),
            size=(H, W),
            mode='bilinear',
            align_corners=False,
        ).squeeze().cpu().numpy()                                 # (H, W)

        cam_up = np.maximum(cam_up, 0)
        if cam_up.max() > 1e-8:
            cam_up /= cam_up.max()

        return cam_up.astype(np.float32)


scorecam_engine = ScoreCAMEngine(model, SCORECAM_LAYER, SCORECAM_BATCH)
print(f'ScoreCAMEngine ready.')
print(f'  Layer  : {SCORECAM_LAYER}')
print(f'  Batch  : {SCORECAM_BATCH} channels/batch')
print(f'  Passes per class per image: 1,664 (= number of denseblock4 channels)')
print(f'  Passes total (500 img × 2 classes avg): ~1,664,000')

## 6. Load Test Images, Predictions and GT Masks

In [ ]:
with open(TEST_JSON) as f:
    test_records = json.load(f)
N_TEST = len(test_records)
print(f'Test records: {N_TEST}')

test_images_raw = np.array([
    np.array(Image.open(TEST_IMG_DIR / rec['filename']).convert('RGB'))
    for rec in tqdm(test_records, desc='Loading test images')
])   # (N_TEST, H, W, 3) uint8
print(f'Test images: {test_images_raw.shape}')


def rebuild_test_masks(
    fold_dirs: Dict[int, Dict[str, Path]],
    n_test: int,
    seed: int = SEED,
) -> np.ndarray:
    """Replay the training notebook's hold-out sampling to recover GT masks.
    n_test is read from len(test_records) to stay in sync automatically.
    """
    rng      = np.random.default_rng(seed)
    per_fold = n_test // 3
    extra    = n_test - 3 * per_fold
    out      = []
    for fold_id, n in [(1, per_fold), (2, per_fold), (3, per_fold + extra)]:
        m_all = np.load(fold_dirs[fold_id]['masks'], mmap_mode='r')
        idx   = rng.choice(len(m_all), size=n, replace=False)
        out.append(np.asarray(m_all[idx], dtype=np.float32))
        del m_all
        gc.collect()
    return np.concatenate(out, axis=0)


print('Rebuilding GT masks...')
test_masks = rebuild_test_masks(FOLD_DIRS, n_test=N_TEST)
print(f'GT masks: {test_masks.shape}')
assert len(test_masks) == N_TEST, (
    f'Mask count {len(test_masks)} != record count {N_TEST}. '
    f'Ensure N_TEST_HOLD in the training notebook matches.'
)

## 7. Run Score-CAM on All Test Images

For each test image we:
1. Read the saved predictions from `test_predictions.json` (classification already done — no re-inference needed)
2. Generate one Score-CAM heatmap per predicted-positive class
3. Compute IoU at all three thresholds against the GT mask
4. Save a triplet figure (original | GT mask | Score-CAM overlay)

> **Memory note:** Score-CAM processes 1,664 masked images in batches of 64.
> Each batch = 64 × 3 × 256 × 256 × 4 bytes ≈ 50 MB GPU memory beyond the model.
> The activations buffer (1,664 × 8 × 8 ≈ 0.4 MB) is negligible.

In [ ]:
scorecam_rows: List[Dict] = []

print(f'Running Score-CAM on {N_TEST} test images...')
print(f'Estimated time: {N_TEST * 2 * 1664 // SCORECAM_BATCH * 0.05 / 60:.0f}–'
      f'{N_TEST * 2 * 1664 // SCORECAM_BATCH * 0.1 / 60:.0f} minutes on GPU')
print()

for img_idx, rec in enumerate(tqdm(test_records, desc='Score-CAM')):

    gt_lbl   = [int(rec['true_labels'][k])  for k in CLASS_KEYS]
    probs    = [float(rec['pred_probs'][k])  for k in CLASS_KEYS]
    pred_lbl = [int(rec['pred_labels'][k])  for k in CLASS_KEYS]
    active   = [i for i, p in enumerate(pred_lbl) if p == 1]

    orig_rgb     = test_images_raw[img_idx]
    mask_6ch     = test_masks[img_idx]
    orig_resized = cv2.resize(orig_rgb, (IMG_SIZE, IMG_SIZE))
    gt_mask_rgb  = render_gt_mask(mask_6ch)

    img_t = preprocess(orig_rgb)   # (1, 3, H, W) on DEVICE

    row: Dict = {
        'image_idx'  : img_idx,
        'filename'   : rec['filename'],
        'cardinality': int(sum(gt_lbl)),
    }
    for j, ck in enumerate(CLASS_KEYS):
        row[f'gt_{ck}']   = gt_lbl[j]
        row[f'pred_{ck}'] = pred_lbl[j]
        row[f'prob_{ck}'] = round(probs[j], 4)

    # ── Generate Score-CAM for each predicted-positive class ─────────────────
    cams_sc: Dict[int, np.ndarray] = {}
    for c_idx in active:
        cam = scorecam_engine.generate(img_t, img_t, c_idx)
        cams_sc[c_idx] = cam

        # IoU at all thresholds
        ck    = CLASS_KEYS[c_idx]
        gt_ch = resize_mask_ch(mask_6ch, c_idx)
        for thr in IOU_THRS:
            row[f'scorecam_iou_{thr:.2f}_{ck}'] = round(
                compute_iou(cam, gt_ch, thr), 4)
        row[f'scorecam_area_{ck}'] = round(
            float((cam >= NOISE_THR).mean()), 4)

    # Fill NaN for non-active classes
    for j, ck in enumerate(CLASS_KEYS):
        for thr in IOU_THRS:
            col = f'scorecam_iou_{thr:.2f}_{ck}'
            if col not in row:
                row[col] = float('nan')
        if f'scorecam_area_{ck}' not in row:
            row[f'scorecam_area_{ck}'] = float('nan')

    scorecam_rows.append(row)

    # ── Save triplet figure for first active class ────────────────────────────
    if active:
        c_show  = active[0]
        overlay = build_cam_overlay(cams_sc[c_show], c_show, orig_resized)
        iou_v   = row.get(f'scorecam_iou_0.50_{CLASS_KEYS[c_show]}', float('nan'))

        fig = plt.figure(figsize=(16, 5.6), facecolor='#0d0d0d')
        gs  = fig.add_gridspec(1, 3, wspace=0.04)
        for col_i, (panel, title) in enumerate([
            (orig_resized, 'Original Image'),
            (gt_mask_rgb,  'Ground-Truth Mask'),
            (overlay,      'Score-CAM Overlay'),
        ]):
            ax = fig.add_subplot(gs[0, col_i])
            ax.imshow(panel)
            ax.set_title(title, color='white', fontsize=10, fontweight='bold', pad=4)
            ax.axis('off')
        r2, g2, b2 = CLASS_COLORS_F32[c_show]
        marker = '\u2713' if bool(gt_lbl[c_show]) else '\u2717'
        iou_s  = f'{iou_v:.3f}' if not (iou_v != iou_v) else 'N/A'
        fig.axes[2].legend(
            handles=[mpatches.Patch(
                facecolor=(r2, g2, b2), edgecolor='white', linewidth=0.4,
                label=f'{marker} {CLASS_DISPLAY[c_show]}  p={probs[c_show]:.2f}  IoU={iou_s}',
            )],
            loc='lower right', fontsize=8, framealpha=0.82,
            facecolor='#111111', edgecolor='#555555', labelcolor='white',
        )
        fig.suptitle(
            f'Image #{img_idx:03d}  |  Score-CAM  |  \u2713=TP  \u2717=FP',
            color='#dddddd', fontsize=10, fontweight='bold', y=1.02,
        )
        plt.savefig(TRIP_DIR / f'scorecam_{img_idx:03d}.png',
                    dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
        plt.close(fig)

    if DEVICE.type == 'cuda' and img_idx % 50 == 0:
        torch.cuda.empty_cache()

scorecam_engine.remove_hooks()

df_sc = pd.DataFrame(scorecam_rows)
df_sc.to_csv(OUT_DIR / 'scorecam_results_test.csv', index=False)
print(f'\nScore-CAM results saved. Shape: {df_sc.shape}')

## 8. Load Gradient-Method Results and Merge

Load the existing `results_50_test.csv` from the main CAM notebook and merge
it with the Score-CAM results on the shared `image_idx` key.
This produces a unified 5-method comparison DataFrame.

In [ ]:
# Load gradient-method results
# Note: the main notebook may have saved this as results_50_test.csv
# regardless of actual test set size — check both filenames
grad_csv_candidates = [
    OUTPUT_ROOT / 'layercam_comparison' / 'results_50_test.csv',
    OUTPUT_ROOT / 'layercam_comparison' / 'results_test.csv',
]
grad_csv = next((p for p in grad_csv_candidates if p.exists()), None)
if grad_csv is None:
    raise FileNotFoundError(
        'Could not find gradient-method results CSV. '
        'Run pannuke_layercam_comparison.ipynb first.'
    )

df_grad = pd.read_csv(grad_csv)
print(f'Gradient results loaded: {df_grad.shape}  from {grad_csv.name}')

# Merge: keep only rows present in both (same image_idx)
# df_sc has image_idx 0..N_TEST-1; df_grad should match
df_all = df_grad.merge(
    df_sc[['image_idx'] + [c for c in df_sc.columns if c.startswith('scorecam')]],
    on='image_idx',
    how='inner',
)
df_all.to_csv(OUT_DIR / 'scorecam_results_500_test.csv', index=False)
print(f'Merged 5-method DataFrame: {df_all.shape}')
print(f'Saved → scorecam_results_500_test.csv')

## 9. IoU Summary — All 5 Methods

In [ ]:
GRAD_METHODS = ['std_gradcam', 'fpn_gradcam', 'std_layercam', 'fpn_layercam']

summary_rows = []
for mkey in ALL_METHODS:
    for thr in IOU_THRS:
        thr_s = f'{thr:.2f}'
        class_means = []
        for ck in CLASS_KEYS:
            col = f'{mkey}_iou_{thr_s}_{ck}'
            if col in df_all.columns:
                class_means.append(df_all[col].dropna().mean())
        summary_rows.append({
            'method'    : METHOD_LABELS[mkey],
            'threshold' : thr,
            'macro_iou' : round(float(np.nanmean(class_means)), 4),
            **{f'iou_{ck}': round(float(v), 4)
               for ck, v in zip(CLASS_KEYS, class_means)},
        })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(OUT_DIR / 'scorecam_iou_summary.csv', index=False)

# Print IoU@0.50 table
df50 = df_summary[df_summary['threshold'] == 0.50]
print(f'IoU@0.50 — All 5 Methods (n={len(df_all)} test images):')
print(f'{"Method":<28}  {"Macro":>7}', end='')
for cd in CLASS_DISPLAY:
    print(f'  {cd[:10]:>10}', end='')
print()
print('-' * 90)
for _, r in df50.iterrows():
    print(f'{r["method"]:<28}  {r["macro_iou"]:>7.4f}', end='')
    for ck in CLASS_KEYS:
        print(f'  {r[f"iou_{ck}"]:>10.4f}', end='')
    print()

# Score-CAM vs best gradient method
sc_macro  = df50[df50['method'] == 'Score-CAM']['macro_iou'].values[0]
best_grad = df50[df50['method'] != 'Score-CAM']['macro_iou'].max()
best_name = df50.loc[df50['macro_iou'] == best_grad, 'method'].values[0]
print(f'\nScore-CAM macro IoU@0.50 : {sc_macro:.4f}')
print(f'Best gradient method      : {best_grad:.4f}  ({best_name})')
print(f'Gap                       : {sc_macro - best_grad:+.4f}')

## 10. Figure 1 — IoU Comparison Bar Chart (All 5 Methods)

In [ ]:
fig, axes = plt.subplots(1, len(IOU_THRS), figsize=(24, 6), sharey=False)
fig.suptitle(
    f'Score-CAM vs GradCAM vs LayerCAM (+ FPN variants) — PanNuke ({N_TEST} test images)\n'
    'Score-CAM is gradient-free; purple bars distinguish it from gradient-based methods',
    fontsize=12, fontweight='bold',
)
x       = np.arange(NUM_CLASSES)
n_m     = len(ALL_METHODS)
w       = 0.15
offsets = np.linspace(-(n_m-1)/2*w, (n_m-1)/2*w, n_m)

for ax, thr in zip(axes, IOU_THRS):
    for mkey, off in zip(ALL_METHODS, offsets):
        means = []
        for ck in CLASS_KEYS:
            col = f'{mkey}_iou_{thr:.2f}_{ck}'
            means.append(df_all[col].dropna().mean() if col in df_all.columns else np.nan)
        edge = '#555555' if mkey == 'scorecam' else 'white'
        lw   = 1.2      if mkey == 'scorecam' else 0.4
        bars = ax.bar(x + off, means, w,
                      label=METHOD_LABELS[mkey],
                      color=METHOD_COLORS[mkey],
                      edgecolor=edge, linewidth=lw, alpha=0.92)
        for b, v in zip(bars, means):
            if not np.isnan(v):
                ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.003,
                        f'{v:.3f}', ha='center', va='bottom',
                        fontsize=4.8, rotation=90)
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_DISPLAY, rotation=20, ha='right', fontsize=8)
    ax.set_title(f'IoU @ {thr:.2f}', fontsize=10)
    ax.set_ylabel('Mean IoU')
    ax.set_ylim(0, 0.38)
    ax.legend(fontsize=6.5, loc='upper right', framealpha=0.6)
    ax.grid(axis='y', alpha=0.25, linestyle=':')

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIG_DIR / f'fig1_iou_5methods.{ext}', bbox_inches='tight', dpi=300)
plt.show()
print('Figure 1 saved.')

## 11. Figure 2 — Macro IoU Heatmap (All 5 Methods × 5 Classes)

In [ ]:
hm_data = df50.set_index('method')[[f'iou_{ck}' for ck in CLASS_KEYS]].astype(float)
hm_data.columns = CLASS_DISPLAY

fig, axes = plt.subplots(1, 2, figsize=(20, 5),
                          gridspec_kw={'width_ratios': [2.5, 1]})
fig.suptitle(f'IoU@0.50 — 5 Methods × 5 Classes  (n={N_TEST})',
             fontsize=12, fontweight='bold')
sns.heatmap(
    hm_data, ax=axes[0],
    cmap='YlOrRd', annot=True, fmt='.4f', annot_kws={'size': 10},
    linewidths=0.5, linecolor='#cccccc',
    cbar_kws={'label': 'Mean IoU@0.50', 'shrink': 0.85},
)
axes[0].tick_params(axis='x', rotation=20, labelsize=9)
axes[0].tick_params(axis='y', rotation=0, labelsize=9)

macros = df50.set_index('method')['macro_iou']
colors = [METHOD_COLORS.get(
    next((k for k, v in METHOD_LABELS.items() if v == m), 'x'), '#AAAAAA')
    for m in macros.index]
axes[1].barh(macros.index, macros.values, color=colors,
             edgecolor='white', linewidth=0.4, alpha=0.92)
for i, v in enumerate(macros.values):
    axes[1].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)
axes[1].set_xlabel('Macro IoU@0.50', fontsize=9)
axes[1].set_xlim(0, max(macros.values) * 1.25)
axes[1].grid(axis='x', alpha=0.25, linestyle=':')
axes[1].axhline(
    y=list(macros.index).index('Score-CAM') - 0.5,
    color='#888888', linestyle='--', linewidth=1, alpha=0.6,
)
axes[1].text(0.001, list(macros.index).index('Score-CAM') - 0.5,
             'gradient-free ▲', va='bottom', fontsize=7, color='#888888')

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIG_DIR / f'fig2_heatmap_5methods.{ext}', bbox_inches='tight', dpi=300)
plt.show()
print('Figure 2 saved.')

## 12. Figure 3 — Score-CAM vs Gradient Methods: IoU@0.50 Scatter

Per-image comparison: Score-CAM IoU vs each gradient method's IoU
for the same (image, class) pair. Points above the diagonal = Score-CAM wins.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5.5), sharey=False)
fig.suptitle(
    'Per-(image, class) IoU@0.50: Score-CAM vs each gradient method\n'
    'Points above diagonal = Score-CAM higher; below = gradient method higher',
    fontsize=11, fontweight='bold',
)
for ax, gkey in zip(axes, GRAD_METHODS):
    sc_vals, gr_vals = [], []
    for ck in CLASS_KEYS:
        sc_col = f'scorecam_iou_0.50_{ck}'
        gr_col = f'{gkey}_iou_0.50_{ck}'
        if sc_col in df_all.columns and gr_col in df_all.columns:
            mask = df_all[sc_col].notna() & df_all[gr_col].notna()
            sc_vals.extend(df_all.loc[mask, sc_col].values)
            gr_vals.extend(df_all.loc[mask, gr_col].values)

    sc_arr = np.array(sc_vals)
    gr_arr = np.array(gr_vals)
    ax.scatter(gr_arr, sc_arr, alpha=0.25, s=8,
               color=METHOD_COLORS[gkey], edgecolors='none')
    lim = max(sc_arr.max(), gr_arr.max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', linewidth=0.8, alpha=0.5)
    win_sc = int((sc_arr > gr_arr).sum())
    win_gr = int((gr_arr > sc_arr).sum())
    n_tot  = len(sc_arr)
    ax.set_xlabel(f'{METHOD_LABELS[gkey]} IoU@0.50', fontsize=9)
    ax.set_ylabel('Score-CAM IoU@0.50', fontsize=9)
    ax.set_title(
        f'{METHOD_LABELS[gkey]}\n'
        f'Score-CAM wins {win_sc}/{n_tot}  |  {METHOD_LABELS[gkey]} wins {win_gr}/{n_tot}',
        fontsize=8.5,
    )
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.grid(alpha=0.2, linestyle=':')
    ax.set_aspect('equal')

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIG_DIR / f'fig3_scatter_scorecam_vs_gradient.{ext}',
                bbox_inches='tight', dpi=300)
plt.show()
print('Figure 3 saved.')

## 13. IoU vs Label Cardinality — Score-CAM vs Gradient Methods

In [ ]:
def pooled_iou_by_card(df: pd.DataFrame, mkey: str, thr: float = 0.50) -> pd.DataFrame:
    thr_s = f'{thr:.2f}'
    rows  = []
    for card in sorted(df['cardinality'].unique()):
        sub  = df[df['cardinality'] == card]
        cols = [f'{mkey}_iou_{thr_s}_{ck}' for ck in CLASS_KEYS
                if f'{mkey}_iou_{thr_s}_{ck}' in df.columns]
        vals = np.concatenate([sub[c].dropna().values for c in cols])
        if len(vals) == 0:
            continue
        rows.append({
            'cardinality': card,
            'mean'       : float(np.nanmean(vals)),
            'sem'        : float(np.nanstd(vals) / max(np.sqrt(len(vals)), 1)),
        })
    return pd.DataFrame(rows)


fig, ax = plt.subplots(figsize=(11, 6))
fig.suptitle(
    f'IoU@0.50 vs Label Cardinality — All 5 Methods (n={len(df_all)})\n'
    'Score-CAM (purple) compared against four gradient-based methods',
    fontsize=12, fontweight='bold',
)

spearman_results = {}
for mkey in ALL_METHODS:
    cd = pooled_iou_by_card(df_all, mkey)
    if len(cd) < 2:
        continue
    rho, p = stats.spearmanr(cd['cardinality'], cd['mean'])
    spearman_results[mkey] = (rho, p)
    lw  = 2.5 if mkey == 'scorecam' else 1.8
    ax.errorbar(
        cd['cardinality'], cd['mean'],
        yerr=cd['sem'] * 1.96,
        fmt=f"{METHOD_MARKERS[mkey]}-",
        color=METHOD_COLORS[mkey],
        linewidth=lw, markersize=9, capsize=5,
        label=f"{METHOD_LABELS[mkey]} (\u03c1={rho:.3f})",
        zorder=3 if mkey == 'scorecam' else 2,
    )

ax.set_xlabel('Label Cardinality', fontsize=11)
ax.set_ylabel('Mean IoU@0.50 (± 95% CI)', fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.25, linestyle=':')
ax.set_xticks(sorted(df_all['cardinality'].unique()))

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIG_DIR / f'fig4_cardinality_5methods.{ext}',
                bbox_inches='tight', dpi=300)
plt.show()

print('Spearman cardinality test:')
for mkey, (rho, p) in spearman_results.items():
    print(f'  {METHOD_LABELS[mkey]:<28}: rho={rho:+.4f}  p={p:.4f}')

## 14. Pairwise Interference — Score-CAM vs Gradient Methods

Extend the interference analysis to Score-CAM using the full dataset.
For fair comparison we run Score-CAM only on the test-set images here
(full-dataset Score-CAM would be ~8.3 M forward passes — future work).

In [ ]:
def build_interference_sc(
    df: pd.DataFrame, mkey: str, thr: float = 0.50, min_n: int = 3,
) -> pd.DataFrame:
    """Pairwise interference for one method on df_all (test images only)."""
    thr_s  = f'{thr:.2f}'
    detail = []
    for c in range(NUM_CLASSES):
        ck     = CLASS_KEYS[c]
        iou_col = f'{mkey}_iou_{thr_s}_{ck}'
        if iou_col not in df.columns:
            continue
        base = df[
            (df[f'gt_{ck}'] == 1) & (df[f'pred_{ck}'] == 1)
        ][[iou_col] + [f'gt_{CLASS_KEYS[cp]}' for cp in range(NUM_CLASSES)]].dropna()

        for cp in range(NUM_CLASSES):
            if c == cp:
                continue
            ck_p = CLASS_KEYS[cp]
            absent  = base[base[f'gt_{ck_p}'] == 0][iou_col].values
            present = base[base[f'gt_{ck_p}'] == 1][iou_col].values
            if len(absent) < min_n or len(present) < min_n:
                continue
            delta   = float(np.nanmean(absent)) - float(np.nanmean(present))
            _, pval = stats.ttest_ind(absent, present, equal_var=False, nan_policy='omit')
            detail.append({
                'method'    : METHOD_LABELS[mkey],
                'class_c'   : CLASS_DISPLAY[c],
                'class_cp'  : CLASS_DISPLAY[cp],
                'delta_iou' : round(delta, 4),
                'n_absent'  : len(absent),
                'n_present' : len(present),
                'p_val'     : round(float(pval), 4),
                'significant': bool(pval < 0.05),
            })
    return pd.DataFrame(detail)


interference_frames = []
for mkey in ALL_METHODS:
    interference_frames.append(build_interference_sc(df_all, mkey))
df_interference = pd.concat(interference_frames, ignore_index=True)
df_interference.to_csv(OUT_DIR / 'scorecam_interference_comparison.csv', index=False)

print('Significant interference pairs per method (p<0.05):')
for mkey in ALL_METHODS:
    sub = df_interference[df_interference['method'] == METHOD_LABELS[mkey]]
    sig = sub['significant'].sum()
    print(f'  {METHOD_LABELS[mkey]:<28}: {sig}/{len(sub)}')

## 15. Final Comparison Summary

In [ ]:
print('=' * 70)
print('SCORE-CAM vs GRADIENT-BASED METHODS — PanNuke')
print('=' * 70)

print(f'\n(1) Macro IoU@0.50 (n={N_TEST} test images):')
for mkey in ALL_METHODS:
    row = df50[df50['method'] == METHOD_LABELS[mkey]]
    if not row.empty:
        family = 'gradient-free' if mkey == 'scorecam' else 'gradient-based'
        print(f'  {METHOD_LABELS[mkey]:<28}: {row["macro_iou"].values[0]:.4f}  [{family}]')

print(f'\n(2) Score-CAM vs gradient-method gaps:')
sc_m = df50[df50['method'] == 'Score-CAM']['macro_iou'].values[0]
for gkey in GRAD_METHODS:
    gm = df50[df50['method'] == METHOD_LABELS[gkey]]['macro_iou'].values[0]
    print(f'  Score-CAM vs {METHOD_LABELS[gkey]:<24}: {sc_m - gm:+.4f}')

print(f'\n(3) Cardinality monotonicity (Spearman rho):')
for mkey, (rho, p) in spearman_results.items():
    print(f'  {METHOD_LABELS[mkey]:<28}: rho={rho:+.4f}  p={p:.4f}')

print(f'\n(4) Interference (significant pairs on test set):')
for mkey in ALL_METHODS:
    sub = df_interference[df_interference['method'] == METHOD_LABELS[mkey]]
    sig = sub['significant'].sum()
    print(f'  {METHOD_LABELS[mkey]:<28}: {sig}/{len(sub)}')

# Save comparison CSV
comp_rows = []
for mkey in ALL_METHODS:
    row50  = df50[df50['method'] == METHOD_LABELS[mkey]]
    rho_p  = spearman_results.get(mkey, (float('nan'), float('nan')))
    sub_int = df_interference[df_interference['method'] == METHOD_LABELS[mkey]]
    comp_rows.append({
        'method'          : METHOD_LABELS[mkey],
        'family'          : 'gradient-free' if mkey == 'scorecam' else 'gradient-based',
        'macro_iou_50'    : row50['macro_iou'].values[0] if not row50.empty else float('nan'),
        'spearman_rho'    : round(rho_p[0], 4),
        'spearman_p'      : round(rho_p[1], 4),
        'n_sig_interference': int(sub_int['significant'].sum()) if not sub_int.empty else 0,
        'n_pairs_tested'  : len(sub_int),
    })
df_comp = pd.DataFrame(comp_rows)
df_comp.to_csv(OUT_DIR / 'scorecam_vs_gradient_comparison.csv', index=False)
print(f'\nComparison CSV saved → scorecam_vs_gradient_comparison.csv')

## 16. Output Summary

In [ ]:
print('=' * 65)
print('OUTPUT FILES')
print('=' * 65)
for f in sorted(OUT_DIR.rglob('*')):
    if f.is_file():
        print(f'  {str(f.relative_to(OUT_DIR)):<55}  {f.stat().st_size/1024:>7.1f} KB')

n_trip = sum(1 for _ in TRIP_DIR.glob('*.png'))
print(f'\nTriplet figures (Score-CAM): {n_trip}')
print(f'\nFPN-ScoreCAM: deferred to future work.')
print('  Reason: 2,432 channels across 3 layers; early-layer masks require')
print('  full downstream propagation — not a simple post-hoc FPN blend.')
print('  Scientific question already addressed by gradient FPN results.')